# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20 10
2. 
Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64
3. 2 2 
4. 
5. [1, 2] 5

In [3]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:     
        
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [5]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [14]:
# 1
print("Строк и столбцов:", df.shape)

print("\nТипы данных:")
print(df.dtypes)

print("\nПропуски delivery_days:")
print(df["delivery_days"].isna().sum())

print("\nУникальные значения:")
for column in ["city", "category", "status", "customer_id"]:
    print(column, df[column].nunique())

Строк и столбцов: (5463, 8)

Типы данных:
order_id           int64
order_date           str
customer_id          str
city                 str
category             str
amount           float64
delivery_days    float64
status               str
dtype: object

Пропуски delivery_days:
160

Уникальные значения:
city 14
category 5
status 2
customer_id 2003


В данных есть 14 вариантов написания 6 городов, 160 пропусков в доставке и тестовый клиент TEST с аномальными заказами. Перед анализом данные необходимо очистить.

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [22]:
# 2
orders = df.loc[paid & not_test, "amount"]

mean_amount = orders.mean()
median_amount = orders.median()
q25 = orders.quantile(0.25)
q75 = orders.quantile(0.75)

print(f"Среднее: {mean_amount:.0f} ₸")
print(f"Медиана: {median_amount:.0f} ₸")
print(f"25-й перцентиль: {q25:.0f} ₸")
print(f"75-й перцентиль: {q75:.0f} ₸")

Среднее: 14672 ₸
Медиана: 10300 ₸
25-й перцентиль: 5880 ₸
75-й перцентиль: 18690 ₸


Типичный чек магазина составляет около 10 300 ₸. Для отчёта лучше использовать медиану, поскольку среднее значение 14 672 ₸ завышается крупными покупками. Половина заказов имеет сумму не более 10 300 ₸.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [29]:
# 3
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]

cancelled = df["status"] == "отменён"

sh_mask = df["city"].isin(SH)
al_mask = df["city"].isin(AL)

all_percent = cancelled.mean() * 100
sh_percent = cancelled[sh_mask].mean() * 100
al_percent = cancelled[al_mask].mean() * 100

print(f"Все заказы: {all_percent:.1f}%")
print(f"Шымкент: {sh_percent:.1f}%")
print(f"Алматы: {al_percent:.1f}%")
print(f"Разница: {sh_percent / al_percent:.2f} раза")

Все заказы: 10.2%
Шымкент: 20.4%
Алматы: 7.9%
Разница: 2.59 раза


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [30]:
# 4
AL = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]

sh = df["city"].isin(SH)
al = df["city"].isin(AL)

sh_sep = df.loc[sh & sep & paid & not_test, "amount"].sum()
sh_oct = df.loc[sh & octo & paid & not_test, "amount"].sum()

al_sep = df.loc[al & sep & paid & not_test, "amount"].sum()
al_oct = df.loc[al & octo & paid & not_test, "amount"].sum()

sh_change = (sh_oct - sh_sep) / sh_sep * 100
al_change = (al_oct - al_sep) / al_sep * 100

print(f"Шымкент сентябрь: {sh_sep:.0f} ₸")
print(f"Шымкент октябрь: {sh_oct:.0f} ₸")
print(f"Изменение: {sh_change:.1f}%")

print()

print(f"Алматы сентябрь: {al_sep:.0f} ₸")
print(f"Алматы октябрь: {al_oct:.0f} ₸")
print(f"Изменение: {al_change:.1f}%")

print(f"\nПадение в Шымкенте сильнее в {abs(sh_change / al_change):.2f} раза")

Шымкент сентябрь: 4949800 ₸
Шымкент октябрь: 4044010 ₸
Изменение: -18.3%

Алматы сентябрь: 15523940 ₸
Алматы октябрь: 14643020 ₸
Изменение: -5.7%

Падение в Шымкенте сильнее в 3.22 раза


Выручка Шымкента снизилась на 18,3%, а Алматы — на 5,7%. Относительное падение в Шымкенте примерно в 3,2 раза сильнее. Менеджеру следует уделить особое внимание снижению продаж в Шымкенте.

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [35]:
# 5
cancel_sh = df[
    df["city"].isin(SH) & (df["status"] == "отменён")
]

print("Отменённых заказов в Шымкенте:", len(cancel_sh))

Отменённых заказов в Шымкенте: 164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [36]:
# 6

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [37]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [38]:
# 8